# Flat 5-class: Rectangle Dice / mAP@0.5 / CPS Grade PCC

기본 flat 5-class와 positive-weighted 모델을 동일 데이터에서 평가합니다.
`CHECKPOINT_NAME`은 기본 `last_model.pt`이며 `best_model.pt`로 변경할 수 있습니다.

**지표 정의**
- Dice: 사각형 내부를 채운 union mask. 클래스별 및 클래스 무관 전체 영역을 계산합니다.
  좌상단 floor, 우하단 ceil로 rasterize하며 영상 범위에 clipping합니다.
  GT와 예측이 모두 빈 경우 NaN으로 기록해 평균에서 제외합니다. 한쪽만 비면 0입니다.
  `dice_pooled`는 전체 타일 intersection과 면적을 합산한 값입니다.
  박스 union이므로 중복 박스/개체 분할을 직접 평가하는 지표는 아닙니다.
- mAP@0.5: 동일 이미지·클래스의 box IoU≥0.5, confidence 순서의 1:1 매칭,
  101 recall 지점의 보간 AP를 클래스 평균합니다. GT 없는 클래스는 NaN으로 제외합니다.
  crowd/ignore/maxDets 제한은 없는 bbox AP50입니다. COCO 공식 평가 전체와 같지는 않습니다.
  AP는 낮은 confidence에서 계산하고, Dice/PCC는 별도 표시용 confidence를 적용합니다.
- PCC: GT와 예측 **CPS Grade**의 Pearson correlation. 기본 단위는 slide입니다.
  먼저 slide 내 세포 수를 합산하고 `100*(Positive Epithelial+Positive Immune)/(Negative Epithelial+Positive Epithelial)`를 계산합니다.
  CPS는 최대 100이며 분모가 0이면 NaN입니다. Epithelial=생존 종양세포, Positive Immune=종양 관련 면역세포라는 라벨 전제의 추정치입니다.
  자동 영역 선택이나 종양 관련성 판정은 수행하지 않습니다.
- 기본 Grade 경계는 비교용 설정 `[1, 5, 10]`: Grade 0(<1), 1(1–<5), 2(5–<10), 3(≥10).
  연구에서 정한 기준이 다르면 `CPS_GRADE_THRESHOLDS`를 바꾸세요. 보편적인 단일 Grade 표준을 뜻하지 않습니다.
- 연속 CPS PCC, CPS MAE, Grade 일치율도 함께 저장합니다. Grade PCC는 서열을 숫자로 코딩한 상관계수입니다.
  표본 수<2 또는 한쪽이 상수면 PCC는 NaN입니다. 계산 불가 쌍의 수와 유효 비율을 함께 보고합니다.
- tile/patch/slide별 결과를 저장합니다. patch/slide CPS는 tile CPS의 평균이 아니라 합산 count에서 계산합니다.
  서로 겹치는 원본 patch가 있으면 slide 합계에 중복 세포가 포함될 수 있으므로 비중첩 평가 영역을 사용하세요.
- CPS 계산 기준: [Agilent 해석 매뉴얼](https://www.agilent.com/cs/library/usermanuals/public/29613-d0039853-22c3-pharmdx-gastric-interpretation-manual-us.pdf).


**기본 평가 데이터:** `new12`는 고정 manifest로 신규 슬라이드 04345~04348의 12개 patch만 선택합니다(48개 tile).
기존 100장과 같은 슬라이드는 포함하지 않으며, 추가 파일이 생겨도 자동 편입하지 않습니다.
새 클래스 `Negative/Positive Tumor(Epithelial)`, `Negative/Positive Immune`, `Other`를
모델의 5개 클래스에 직접 대응합니다. 기존 7-class 이름도 5-class로 합쳐 읽습니다.
알 수 없는 클래스명은 오류로 알리고, 기존의 비정상/100px 초과 bbox 필터는 유지하며 제외 수를 CSV로 남깁니다.
`all`/`balanced_val`은 기존 학습 데이터를 포함합니다. 신규 12장도 재학습에 포함했다면 독립 테스트가 아닙니다.
CPS 계산 불가 슬라이드(종양세포 분모 0)는 별도 보고하므로 Grade PCC의 유효 표본 수를 확인하세요.

참고: [AP 개념](https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision),
[Pearson correlation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html).

## 전체 vs other 제외 비교

`all5`와 `without_other`를 같은 추론 결과에서 나란히 계산합니다.
- Dice union: 전체 박스 영역 vs other 박스를 제외한 영역. Union Dice는 클래스 무관 영역 점수입니다.
- Dice class macro: 5개 클래스 vs other 제외 4개 클래스의 pooled Dice 평균.
- mAP@0.5: 전체 클래스 AP 평균 vs other 제외 AP 평균(GT 없는 클래스는 제외).
- CPS Grade PCC: other를 사용하지 않으므로 두 scope에서 동일합니다.

NMS 전 other 예측을 제거하거나 재분류하지 않습니다. 실제 관심 세포를 other로 예측하면
관심 클래스 AP에서는 FN, other 위치를 관심 클래스로 예측하면 FP로 남습니다.
`comparison_with_without_other.csv`와 `with_without_other.png`에 비교 결과를 저장합니다.

## GT 개수에 따른 평가 패치 선택

원본 패치에서 유효 GT가 10개 미만이면 제외합니다(0~9개 제외, 10개 이상 포함).
기본 `PATCH_GT_INCLUDE_OTHER=True`는 other를 포함한 개수입니다. False로 바꾸면
other를 제외한 관심 클래스 GT 수로 선택합니다. 두 모델과 두 지표 scope는 같은 패치를 사용합니다.
선택 후 4개 타일로 나누므로 개별 타일의 GT가 10개 미만이어도 평가에 포함됩니다.
`patch_selection.csv`와 `excluded_patches.csv`에 선택 근거를 저장합니다.
CPS도 선택된 패치만 합산합니다. 필터 적용 결과는 전체 패치에 대한 성능과 구분해서 해석하세요.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm
from nets import nn
from utils import util
from utils.ihc_dataset import IHCHierarchicalDataset, discover_records, split_records_by_slide_balanced, slide_id_from_record
from utils.flat5_eval_dataset import Flat5EvaluationDataset, select_fixed_test_records
from utils.rectangle_metrics import clip_boxes, rectangle_mask, rectangle_dice, average_precision_50, cps_from_flat5_counts, cps_grade, paired_score_correlation

GPU_ID = 0
CHECKPOINT_NAME = 'last_model.pt'
MODEL_ROOT = Path('../../model/mediautoAI_PDL1_Stomach')
CHECKPOINTS = {
    'flat5': MODEL_ROOT / 'pdl1_flat_5class' / CHECKPOINT_NAME,
    'flat5_positive_weighted': MODEL_ROOT / 'pdl1_flat_5class_positive_weighted' / CHECKPOINT_NAME,
}
EVAL_MODE = 'new12'  # new12 / all / balanced_val / external
NEW_TEST_MANIFEST = Path('data_manifests/pdl1_new_12_test.json')
DATA_ROOT = Path('../../data/mediautoAI_PDL1_Stomach')
EXTERNAL_DATA_ROOT = None
INPUT_SIZE, RESIZE_SIZE = 512, 1024
AP_CONFIDENCE_THRESHOLD = 0.001
OPERATING_CONFIDENCE = 0.10  # Dice, CPS Grade PCC, 시각화에 사용
CPS_GRADE_THRESHOLDS = (1, 5, 10)  # 0:<1, 1:1–<5, 2:5–<10, 3:>=10
CPS_SUMMARY_UNIT = 'slide'  # tile / patch / slide
if CPS_SUMMARY_UNIT not in ('tile', 'patch', 'slide'):
    raise ValueError('Invalid CPS_SUMMARY_UNIT')
cps_grade(0, CPS_GRADE_THRESHOLDS)  # validate grade cutoffs
NMS_IOU = 0.45
MIN_GT_PER_PATCH = 10  # 원본 patch의 유효 GT 수; 10개는 포함
PATCH_GT_INCLUDE_OTHER = True  # False: other를 제외한 GT 수로 패치 선택
MAX_TILES = None  # None이면 전체. 동작 확인 시 2 등으로 설정.
VIS_INDICES = [0, 12, 24, 36]  # each of the four new slides
SHOW_FIGURES = True
CLASS_NAMES = {0: 'Negative Epithelial', 1: 'Negative Immune', 2: 'Positive Epithelial', 3: 'Positive Immune', 4: 'other'}
SOURCE_TO_FLAT = np.array([0, 1, 1, 2, 3, 3, 4])
OUTPUT_DIR = Path('results') / ('flat5_dice_map_pcc_' + datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f'))
if not 0 <= AP_CONFIDENCE_THRESHOLD <= OPERATING_CONFIDENCE <= 1:
    raise ValueError('Require 0 <= AP threshold <= operating threshold <= 1')
if torch.cuda.is_available():
    if not 0 <= GPU_ID < torch.cuda.device_count():
        raise ValueError('Invalid GPU_ID')
    torch.cuda.set_device(GPU_ID)
    device = torch.device(f'cuda:{GPU_ID}')
else:
    device = torch.device('cpu')
for path in CHECKPOINTS.values():
    if not path.is_file(): raise FileNotFoundError(path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print(device, OUTPUT_DIR.resolve())


In [ ]:
if EVAL_MODE == 'new12':
    records = select_fixed_test_records(discover_records(DATA_ROOT), DATA_ROOT, NEW_TEST_MANIFEST)
    print('Fixed new test set: 12 patches; excludes previous 100 patches and their slides.')
elif EVAL_MODE == 'external':
    if EXTERNAL_DATA_ROOT is None: raise ValueError('Set EXTERNAL_DATA_ROOT')
    if Path(EXTERNAL_DATA_ROOT).resolve() == DATA_ROOT.resolve():
        raise ValueError('External evaluation requires a different data directory')
    records = discover_records(EXTERNAL_DATA_ROOT)
else:
    records = discover_records(DATA_ROOT)
    if EVAL_MODE == 'balanced_val':
        _, records = split_records_by_slide_balanced(records, val_slide_count=3)
    elif EVAL_MODE != 'all': raise ValueError(EVAL_MODE)
    print('주의: 학습에 사용한 데이터 평가입니다. 일반화 성능으로 해석하지 마세요.')
if not records: raise ValueError('No evaluation records')
if not isinstance(MIN_GT_PER_PATCH, int) or MIN_GT_PER_PATCH < 1:
    raise ValueError('MIN_GT_PER_PATCH must be a positive integer')
# Apply one GT-only source-patch filter to both models and both metric scopes.
# Count labels after invalid/oversized bbox filtering, before four-crop tiling.
candidate_records = records
selection_rows, records = [], []
for image_path, label_path in candidate_records:
    labels = Flat5EvaluationDataset.load_labels(label_path)
    total_gt = len(labels)
    non_other_gt = int((labels[:, 0] != 4).sum())
    selection_gt = total_gt if PATCH_GT_INCLUDE_OTHER else non_other_gt
    included = selection_gt >= MIN_GT_PER_PATCH
    selection_rows.append({'patch': str(image_path),
        'slide': slide_id_from_record((image_path, label_path)),
        'gt_all': total_gt, 'gt_without_other': non_other_gt,
        'selection_gt': selection_gt, 'included': included,
        'reason': '' if included else ('empty_gt' if selection_gt == 0 else 'gt_below_minimum')})
    if included:
        records.append((image_path, label_path))
patch_selection = pd.DataFrame(selection_rows)
patch_selection.to_csv(OUTPUT_DIR / 'patch_selection.csv', index=False)
excluded_patches = patch_selection[~patch_selection.included]
excluded_patches.to_csv(OUTPUT_DIR / 'excluded_patches.csv', index=False)
print(f'GT filter: {len(candidate_records)} candidates -> {len(records)} included; '
      f'{len(excluded_patches)} excluded (minimum={MIN_GT_PER_PATCH}, '
      f'include_other={PATCH_GT_INCLUDE_OTHER})')
if len(excluded_patches):
    display(excluded_patches)
if not records:
    raise ValueError('No patches remain after GT filtering; see patch_selection.csv')
dataset = Flat5EvaluationDataset(records, input_size=INPUT_SIZE, resize_size=RESIZE_SIZE,
    augment=False, random_crop=False, four_crop=True, cache_images=True)
if MAX_TILES is not None and MAX_TILES <= 0: raise ValueError('MAX_TILES must be positive')
tile_indices = list(range(len(dataset)))[:MAX_TILES]
manifest = pd.DataFrame([{'tile': i, 'patch': str(records[i//4][0]),
    'slide': slide_id_from_record(records[i//4]), 'quadrant': i%4} for i in tile_indices])
manifest.to_csv(OUTPUT_DIR / 'evaluation_tiles.csv', index=False)
print(f'{len(records)} source patches / {len(tile_indices)} evaluated tiles')
if MAX_TILES is not None:
    print('Subset run: patch/slide counts may cover only some quadrants or patches.')
display(manifest.head())

# Audit native five-class labels before evaluation; record existing box exclusions.
audit_rows = []
for record_index, (image_path, label_path) in enumerate(records):
    labels = dataset._get_resized_pair(record_index)[1]
    raw_count = len(json.loads(Path(label_path).read_text())['cells'])
    row = {'image':str(image_path), 'slide':slide_id_from_record((image_path,label_path)),
           'raw_cells':raw_count, 'loaded_cells':len(labels), 'excluded_boxes':raw_count-len(labels)}
    row.update({name:int((labels[:,0]==cls).sum()) for cls,name in CLASS_NAMES.items()})
    audit_rows.append(row)
label_audit = pd.DataFrame(audit_rows)
label_audit.to_csv(OUTPUT_DIR/'label_loading_audit.csv',index=False)
display(label_audit.groupby('slide')[list(CLASS_NAMES.values())].sum())


In [ ]:
def load_model(path):
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    if checkpoint.get('num_classes') != 5 or checkpoint.get('architecture') != 'pdl1_flat_5class':
        raise ValueError(f'Expected flat 5-class checkpoint: {path}')
    names = checkpoint.get('class_names', checkpoint.get('params', {}).get('names'))
    if names is None or {int(k): v for k,v in names.items()} != CLASS_NAMES:
        raise ValueError('Checkpoint class order differs from this notebook')
    if tuple(checkpoint.get('source_to_flat', SOURCE_TO_FLAT)) != tuple(SOURCE_TO_FLAT):
        raise ValueError('Checkpoint label mapping differs')
    model = nn.yolo_v11_m(num_classes=5)
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    metadata = {'checkpoint': str(path.resolve()), 'epoch': int(checkpoint['epoch'])+1,
        'cls_pos_weight': checkpoint.get('params', {}).get('cls_pos_weight')}
    del checkpoint
    return model.to(device).eval(), metadata

def gt_for_tile(index):
    image, labels, boxes, _ = dataset[index]
    xywh = boxes.numpy() * INPUT_SIZE
    xyxy = np.concatenate((xywh[:,:2]-xywh[:,2:]/2, xywh[:,:2]+xywh[:,2:]/2), axis=1)
    xyxy, keep = clip_boxes(xyxy, INPUT_SIZE, INPUT_SIZE)
    return image, xyxy[keep], labels.numpy().astype(int)[keep]  # dataset already outputs flat-five IDs

def save_mask_preview(image, gt_boxes, pred_boxes, path, title):
    gt = rectangle_mask(gt_boxes, INPUT_SIZE, INPUT_SIZE)
    pred = rectangle_mask(pred_boxes, INPUT_SIZE, INPUT_SIZE)
    overlay = np.zeros((INPUT_SIZE, INPUT_SIZE, 3), dtype=np.uint8)
    overlay[gt & pred] = [255, 255, 255]
    overlay[gt & ~pred] = [0, 200, 255]
    overlay[pred & ~gt] = [255, 80, 80]
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for axis, content, label in zip(axes, [image.permute(1,2,0).numpy(), gt, pred, overlay],
            ['Image', 'GT rectangle union', 'Prediction rectangle union', 'White=intersection; cyan=GT only; red=pred only']):
        axis.imshow(content, cmap='gray', vmin=0, vmax=1 if content.ndim==2 else 255)
        axis.set_title(label, fontsize=9); axis.axis('off')
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=140)
    if SHOW_FIGURES: plt.show()
    plt.close(fig)


In [ ]:
dice_rows, count_rows, ap_rows, checkpoint_rows = [], [], [], []
map_by_model = {}
for model_name, path in CHECKPOINTS.items():
    model, metadata = load_model(path)
    checkpoint_rows.append({'model': model_name, **metadata})
    ap_records, export_predictions = [], []
    # Batch size 1 avoids the existing NMS batch timeout skipping later images.
    with torch.inference_mode():
        for index in tqdm(tile_indices, desc=model_name):
            image, gt_boxes, gt_labels = gt_for_tile(index)
            images = image.unsqueeze(0).to(device).float()/255
            with torch.amp.autocast('cuda', enabled=device.type=='cuda'):
                raw = model(images)
            pred = util.non_max_suppression(raw.float(), confidence_threshold=AP_CONFIDENCE_THRESHOLD,
                iou_threshold=NMS_IOU, class_agnostic=True)[0].cpu().numpy()
            pred_boxes, valid = clip_boxes(pred[:,:4], INPUT_SIZE, INPUT_SIZE)
            pred = pred[valid]; pred_boxes = pred_boxes[valid]
            ap_records.append({'gt_boxes': gt_boxes, 'gt_labels': gt_labels,
                'pred_boxes': pred_boxes, 'pred_labels': pred[:,5].astype(int), 'pred_scores': pred[:,4]})
            export_predictions.append({'tile': index, 'gt_boxes': gt_boxes.tolist(), 'gt_labels': gt_labels.tolist(),
                'pred_boxes': pred_boxes.tolist(), 'pred_labels': pred[:,5].astype(int).tolist(), 'pred_scores': pred[:,4].tolist()})
            selected = pred[:,4] > OPERATING_CONFIDENCE
            boxes, labels = pred_boxes[selected], pred[selected,5].astype(int)
            ids = {'model':model_name, 'tile':index, 'patch':str(records[index//4][0]), 'slide':slide_id_from_record(records[index//4])}
            for cls, name in [(-1, 'ALL')] + list(CLASS_NAMES.items()):
                gt_cls = gt_boxes if cls==-1 else gt_boxes[gt_labels==cls]
                pred_cls = boxes if cls==-1 else boxes[labels==cls]
                dice_rows.append({**ids, 'class':name, **rectangle_dice(gt_cls, pred_cls, INPUT_SIZE, INPUT_SIZE)})
                count_rows.append({**ids, 'class':name, 'gt_count':len(gt_cls), 'pred_count':len(pred_cls)})
            dice_rows.append({**ids, 'class':'NON_OTHER',
                **rectangle_dice(gt_boxes[gt_labels!=4], boxes[labels!=4], INPUT_SIZE, INPUT_SIZE)})
            if index in VIS_INDICES:
                save_mask_preview(image, gt_boxes, boxes, OUTPUT_DIR/f'{model_name}_tile_{index:03d}_dice.png',
                    f'{model_name} | tile {index} | class-agnostic rectangle masks')
                save_mask_preview(image, gt_boxes[gt_labels!=4], boxes[labels!=4],
                    OUTPUT_DIR/f'{model_name}_tile_{index:03d}_dice_no_other.png',
                    f'{model_name} | tile {index} | excluding other')
    rows, map_by_model[model_name] = average_precision_50(ap_records, CLASS_NAMES)
    ap_rows.extend({'model':model_name, 'class':CLASS_NAMES[row['class_id']], **row} for row in rows)
    (OUTPUT_DIR/f'{model_name}_boxes.json').write_text(json.dumps(export_predictions))
    del model, ap_records
    gc.collect()
    if device.type=='cuda': torch.cuda.empty_cache()

dice_table, counts, ap_table = pd.DataFrame(dice_rows), pd.DataFrame(count_rows), pd.DataFrame(ap_rows)


In [ ]:
dice_summary_rows = []
for (model_name, name), group in dice_table.groupby(['model','class'], sort=False):
    denominator = group.gt_area.sum() + group.pred_area.sum()
    dice_summary_rows.append({'model':model_name, 'class':name,
        'dice_pooled':2*group.intersection.sum()/denominator if denominator else np.nan,
        'dice_mean_tile':group.dice.mean(), 'nonempty_tiles':int(group.dice.notna().sum()),
        'both_empty_tiles':int(group.dice.isna().sum())})
dice_summary = pd.DataFrame(dice_summary_rows)
cps_rows = []
for unit in ['tile','patch','slide']:
    grouped = counts[counts['class']!='ALL'].groupby(['model',unit,'class'], sort=False)[['gt_count','pred_count']].sum().reset_index()
    grouped.to_csv(OUTPUT_DIR/f'counts_{unit}.csv',index=False)
    for (model_name, sample), group in grouped.groupby(['model',unit], sort=False):
        group = group.set_index('class').reindex(list(CLASS_NAMES.values()), fill_value=0)
        gt = cps_from_flat5_counts(group.gt_count.to_numpy())
        pred = cps_from_flat5_counts(group.pred_count.to_numpy())
        cps_rows.append({'model':model_name, 'unit':unit, 'sample':sample,
            **{f'gt_{k}':v for k,v in gt.items()}, **{f'pred_{k}':v for k,v in pred.items()},
            'gt_grade':cps_grade(gt['cps'], CPS_GRADE_THRESHOLDS),
            'pred_grade':cps_grade(pred['cps'], CPS_GRADE_THRESHOLDS)})
cps_table = pd.DataFrame(cps_rows)
pcc_rows = []
for (model_name, unit), group in cps_table.groupby(['model','unit'], sort=False):
    grade = paired_score_correlation(group.gt_grade.to_numpy(),group.pred_grade.to_numpy())
    score = paired_score_correlation(group.gt_cps.to_numpy(),group.pred_cps.to_numpy())
    valid = np.isfinite(group.gt_grade) & np.isfinite(group.pred_grade)
    pcc_rows.append({'model':model_name,'unit':unit,
        'pcc_cps_grade':grade['pcc'], 'pcc_cps':score['pcc'],
        'grade_accuracy':float((group.loc[valid,'gt_grade']==group.loc[valid,'pred_grade']).mean()) if valid.any() else np.nan,
        'grade_mae':grade['mae'],'cps_mae':score['mae'],
        'n_pairs':grade['n_pairs'],'n_valid_pairs':grade['n_valid_pairs'],
        'n_invalid_gt':grade['n_invalid_gt'],'n_invalid_pred':grade['n_invalid_pred'],
        'valid_pair_fraction':grade['valid_pair_fraction'],
        'grade_pcc_undefined_reason':grade['pcc_undefined_reason'],
        'cps_pcc_undefined_reason':score['pcc_undefined_reason']})
pcc_table = pd.DataFrame(pcc_rows)
summary_rows=[]
for model_name in CHECKPOINTS:
    ds=dice_summary[dice_summary.model==model_name]
    ap=ap_table[ap_table.model==model_name]
    pc=pcc_table[(pcc_table.model==model_name)&(pcc_table.unit==CPS_SUMMARY_UNIT)].iloc[0]
    for scope, selected_ids, union_name in [
        ('all5', list(CLASS_NAMES), 'ALL'),
        ('without_other', [c for c in CLASS_NAMES if c!=4], 'NON_OTHER'),
    ]:
        selected_names=[CLASS_NAMES[c] for c in selected_ids]
        selected_ap=ap[ap.class_id.isin(selected_ids)]
        union=ds[ds['class']==union_name].iloc[0]
        summary_rows.append({'model':model_name,'scope':scope,
            'map50':selected_ap.ap50.mean(),
            'map_classes_with_gt':int(selected_ap.ap50.notna().sum()),
            'dice_union_pooled':union.dice_pooled,
            'dice_union_mean_tile':union.dice_mean_tile,
            'dice_class_macro_pooled':ds[ds['class'].isin(selected_names)].dice_pooled.mean(),
            'cps_unit':CPS_SUMMARY_UNIT,'pcc_cps_grade':pc.pcc_cps_grade,'pcc_cps':pc.pcc_cps,
            'grade_accuracy':pc.grade_accuracy,'cps_mae':pc.cps_mae,'n_valid_pairs':pc.n_valid_pairs,
            'valid_pair_fraction':pc.valid_pair_fraction})
summary=pd.DataFrame(summary_rows)
comparison=summary.pivot(index='model',columns='scope',values=['dice_union_pooled','dice_class_macro_pooled','map50'])
comparison.to_csv(OUTPUT_DIR/'comparison_with_without_other.csv')
print('Side-by-side: same predictions/NMS; other excluded only during metric aggregation')
display(comparison.round(4))
print('CPS Grade PCC is common to both scopes (other is not used in CPS).')
for filename, table in [('summary',summary),('dice_per_tile',dice_table),('dice_per_class',dice_summary),
                        ('ap50_per_class',ap_table),('pcc_cps_grade',pcc_table),('cps_by_sample',cps_table),('checkpoints',pd.DataFrame(checkpoint_rows))]:
    table.to_csv(OUTPUT_DIR/f'{filename}.csv',index=False)
settings={'min_gt_per_patch':MIN_GT_PER_PATCH,'patch_gt_include_other':PATCH_GT_INCLUDE_OTHER,
    'candidate_patches':len(candidate_records),'included_patches':len(records),
    'excluded_patches':len(excluded_patches),'patch_filter_unit':'source patch before tiling',
    'eval_mode':EVAL_MODE,'label_schema':'native flat5 (legacy names supported)',
    'test_manifest':str(NEW_TEST_MANIFEST) if EVAL_MODE=='new12' else None,'data_root':str(EXTERNAL_DATA_ROOT if EVAL_MODE=='external' else DATA_ROOT),
    'ap_confidence':AP_CONFIDENCE_THRESHOLD,'operating_confidence':OPERATING_CONFIDENCE,
    'nms_iou':NMS_IOU,'ap_iou':0.5,'ap_method':'101-point interpolated; no maxDets/crowd/ignore',
    'evaluation_scopes':['all5','without_other'],
    'other_policy':'retain original class-agnostic NMS; exclude other from masks and AP averaging only',
    'dice_empty_policy':'NaN if both empty','pcc_definition':'GT vs predicted ordinal CPS Grade',
    'cps_grade_thresholds':list(CPS_GRADE_THRESHOLDS),'cps_summary_unit':CPS_SUMMARY_UNIT,
    'cps_cap':100,'cps_zero_denominator':'NaN; excluded pair with coverage reported',
    'checkpoint_name':CHECKPOINT_NAME,'max_tiles':MAX_TILES,'input_size':INPUT_SIZE,'resize_size':RESIZE_SIZE}
(OUTPUT_DIR/'settings.json').write_text(json.dumps(settings,ensure_ascii=False,indent=2))
display(summary.round(4)); display(dice_summary.round(4)); display(ap_table.round(4)); display(pcc_table.round(4))


In [ ]:
fig,axes=plt.subplots(len(CHECKPOINTS),2,figsize=(12,5*len(CHECKPOINTS)),squeeze=False)
for row,model_name in enumerate(CHECKPOINTS):
    group=cps_table[(cps_table.model==model_name)&(cps_table.unit==CPS_SUMMARY_UNIT)]
    stats=pcc_table[(pcc_table.model==model_name)&(pcc_table.unit==CPS_SUMMARY_UNIT)].iloc[0]
    valid=group.dropna(subset=['gt_cps','pred_cps'])
    axis=axes[row,0]
    axis.scatter(valid.gt_cps,valid.pred_cps,s=35,alpha=.7)
    axis.plot([0,100],[0,100],'k--',linewidth=1)
    axis.set(xlim=(-2,102),ylim=(-2,102),xlabel='GT CPS',ylabel='Predicted CPS',
             title=f'{model_name} | {CPS_SUMMARY_UNIT} CPS PCC={stats.pcc_cps:.3f}')
    axis=axes[row,1]
    pairs=valid.groupby(['gt_grade','pred_grade']).size().reset_index(name='n')
    axis.scatter(pairs.gt_grade,pairs.pred_grade,s=pairs.n*55,alpha=.6)
    for _,pair in pairs.iterrows():
        axis.annotate(str(int(pair.n)),(pair.gt_grade,pair.pred_grade),ha='center',va='center',fontsize=9)
    maximum=len(CPS_GRADE_THRESHOLDS)
    axis.plot([0,maximum],[0,maximum],'k--',linewidth=1)
    axis.set(xticks=range(maximum+1),yticks=range(maximum+1),xlim=(-.5,maximum+.5),ylim=(-.5,maximum+.5),
        xlabel='GT CPS Grade',ylabel='Predicted CPS Grade',
        title=f'{model_name} | Grade PCC={stats.pcc_cps_grade:.3f} | valid={int(stats.n_valid_pairs)}/{int(stats.n_pairs)}')
fig.tight_layout();fig.savefig(OUTPUT_DIR/f'pcc_cps_grade_{CPS_SUMMARY_UNIT}.png',dpi=150)
if SHOW_FIGURES:plt.show()
plt.close(fig)
print(f'Saved results: {OUTPUT_DIR.resolve()}')


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,5))
for axis,metric,title in zip(axes,
        ['dice_union_pooled','dice_class_macro_pooled','map50'],
        ['Rectangle union Dice (class-agnostic)','Mean per-class pooled Dice','mAP@0.5']):
    table=summary.pivot(index='model',columns='scope',values=metric).reindex(columns=['all5','without_other'])
    table.plot.bar(ax=axis,ylim=(0,1),rot=10)
    axis.set_title(title);axis.set_xlabel('');axis.grid(axis='y',alpha=.3)
fig.tight_layout();fig.savefig(OUTPUT_DIR/'with_without_other.png',dpi=150)
if SHOW_FIGURES:plt.show()
plt.close(fig)
print(f'Results: {OUTPUT_DIR.resolve()}')
